In [1]:
import pandas as pd
from src.data import DATA_DIR_INTERIM
from src.config import LLM_NAMES

from src.io import load_qrels_from_path, read_metadata
from topic_gen.evaluate import MetaExperiment
from topic_gen.evaluate.io import load_from_irds
from topic_gen.evaluate.measures_agreement import CohenKappa, AreaUnderReceiver, MeanAverageError
from topic_gen.evaluate.measures_agreement_multiple import LabelDistribution

from topic_gen.evaluate.utils import QrelsTransformer

from topic_gen import logger
logger.setLevel("DEBUG")

# Prerequisite
#### Can we reproduce the qrels quality by Thomas et al. with open models?
Yes! Open models outperform the results on GPT 4.1

In [2]:
BASE_DIR = DATA_DIR_INTERIM / "robust" / "qrels-reference"
experiments = load_qrels_from_path(
    BASE_DIR, binarize_qrels=1, drop_relevance_values=999)

[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-11-30_11:01:37 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-12-19_09:46:06 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-11-30_12:12:38 is empty after processing, skipping...


Error loading experiment from 2025-12-19_14:05:21: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/robust/qrels-reference/2025-12-19_14:05:21/qrels.csv.gz'


In [22]:
baseline = load_from_irds("disks45/nocr/trec-robust-2004", binarize_qrels=1)

In [23]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), AreaUnderReceiver(),
              MeanAverageError(), LabelDistribution()],
    filter_qrels=True
)

In [24]:
res = meta_exp.evaluate()

In [25]:
metadata = read_metadata(BASE_DIR)

[topic_gen] [WARNING] (io.py:48) Metadata not found for result 2025-12-08_07:09:33, skipping...


In [43]:
df = pd.DataFrame(res)
missing = df.groupby("name")["missing_qrels"].max().to_dict()  # we take the max per group because label dist does not report missings yet

df = df.pivot(index="name", columns="measure", values="value").reset_index()

df = df.merge(metadata, left_on="name", right_on="date")
df["missing"] = df["name"].map(missing)
df["missing"] = abs(df["missing"]- 311410 + 2951)


In [44]:
df[["name", "model", "prompt", "CohenKappa", "MeanAverageError",
    "AreaUnderReceiver", "label_dist(0)", "label_dist(1)", "missing"]].round(2)

,name,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver,label_dist(0),label_dist(1),missing
0,2025-11-26_11:17:24,gpt-4.1,-DNA-zero-shot,0.64,0.17,0.81,0.43,0.57,0
1,2025-11-27_06:56:22,Qwen3-30B,-DNA-zero-shot,0.76,0.10,0.90,0.28,0.72,21
2,2025-11-27_07:02:05,Qwen3-30B,-DNA-zero-shot-masked-description-narrative,0.68,0.13,0.86,0.29,0.71,15
3,2025-11-27_07:07:40,Qwen3-30B,-DNA-zero-shot-masked-description,0.73,0.12,0.88,0.29,0.71,19
4,2025-11-27_07:13:24,Qwen3-30B,-DNA-zero-shot-masked-narrative,0.73,0.11,0.89,0.28,0.72,17
5,2025-11-27_07:18:57,Qwen3-30B,-DNA-zero-shot-masked-title-description,0.66,0.15,0.85,0.28,0.72,39
6,2025-11-27_07:24:37,Qwen3-30B,-DNA-zero-shot-masked-title-narrative,0.72,0.12,0.88,0.29,0.71,18
7,2025-11-27_07:30:10,Qwen3-30B,-DNA-zero-shot-masked-title,0.79,0.09,0.91,0.30,0.70,36
8,2025-11-28_13:57:38,GPT-OSS-120B,-DNA-zero-shot,0.71,0.14,0.84,0.43,0.57,315
9,2025-11-28_15:17:58,GPT-OSS-20B,-DNA-zero-shot,0.64,0.18,0.82,0.48,0.52,8


In [9]:
def to_table(df: pd.DataFrame) -> pd.DataFrame:
    table = df.copy()
    table = table.round(2)
    table = table[
        ["model", "CohenKappa", "MeanAverageError", "AreaUnderReceiver",
            "label_dist(0)", "label_dist(1)", "missing"]
    ]
    table.rename(
        columns={
            "CohenKappa": "Cohen $\kappa$",
            "MeanAverageError": "MAE",
            "AreaUnderReceiver": "AUC",
            "label_dist(0)": "0",
            "label_dist(1)": "1",
            "model": "Model",
            "missing": "Missing",
        },
        inplace=True,
    )

    table["Model"].replace(
        {
            "qwen3-14B-no-think": "Qwen3-14B",
            "qwen3-30B-no-think": "Qwen3-30B",
            "deepseek-V3.2": "DeepSeek V3.2",
            "gpt-4.1": "GPT-4.1",
            "gpt-oss-120B": "GPT-OSS-120B",
            "gpt-oss-20B": "GPT-OSS-20B",
            "llama3-1-8B-instruct": "LLaMA-3.1-8B-instruct",
            "llama3-1-70B_instruct_q8_0_MT1000_ollama": "LLaMA-3.1-70B-instruct",
        },
        inplace=True,
    )
    formatters = {
        "Cohen $\\kappa$": lambda x: f"{x:.2f}",
        "MAE": lambda x: f"{x:.2f}",
        "AUC": lambda x: f"{x:.2f}",
        "0": lambda x: f"{x:.2f}",
        "1": lambda x: f"{x:.2f}",
    }
    print(table.to_latex(index=False, formatters=formatters))

In [10]:
to_table(df)

/tmp/ipykernel_3980884/1845165292.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  table["Model"].replace(


\begin{tabular}{lrrrrrr}
\toprule
Model & Cohen $\kappa$ & MAE & AUC & 0 & 1 & Missing \\
\midrule
GPT-4.1 & 0.64 & 0.17 & 0.81 & 0.43 & 0.57 & 0 \\
Qwen3-30B & 0.76 & 0.10 & 0.90 & 0.28 & 0.72 & 0 \\
GPT-OSS-120B & 0.71 & 0.14 & 0.84 & 0.43 & 0.57 & 0 \\
GPT-OSS-20B & 0.64 & 0.18 & 0.82 & 0.48 & 0.52 & 0 \\
Llama3.1-8B & 0.63 & 0.16 & 0.82 & 0.33 & 0.67 & 0 \\
GPT-OSS-120B-O & 0.72 & 0.14 & 0.85 & 0.43 & 0.57 & 0 \\
Llama3.1-70B & 0.77 & 0.10 & 0.88 & 0.34 & 0.66 & 0 \\
Deepseek-V3.2 & 0.72 & 0.13 & 0.85 & 0.42 & 0.58 & 0 \\
Qwen3-14B & 0.72 & 0.13 & 0.85 & 0.39 & 0.61 & 0 \\
Qwen3-Next-80B & 0.77 & 0.11 & 0.88 & 0.35 & 0.65 & 0 \\
Llama3.3-70B & 0.80 & 0.09 & 0.91 & 0.30 & 0.70 & 0 \\
\bottomrule
\end{tabular}



### Do LLM judges benefit from additional information in the topic?
The original TREC topics are judged by an LLM but one or two topic components are masked. For example, the prompt `-DNA-zero-shot-masked-title` only contains the **description** and **narrative** of the TREC topic. 

- Generally the differences are low
- Masking the title outperforms the full topic
- More context seems to be better

In [45]:
BASE_DIR = DATA_DIR_INTERIM / "robust" / "qrels-topics-masked"
experiments = load_qrels_from_path(
    BASE_DIR, binarize_qrels=1, drop_relevance_values=999)

[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-11-30_11:01:37 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-12-08_07:00:48 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-12-19_09:46:06 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-11-30_12:12:38 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-12-08_07:07:17 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-12-08_07:09:21 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:109) Qrels for result 2025-12-09_01:00:54 is empty after processing, skipping...


Error loading experiment from 2025-12-08_07:09:33: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/robust/qrels-topics-masked/2025-12-08_07:09:33/qrels.csv.gz'


In [46]:
baseline = load_from_irds("disks45/nocr/trec-robust-2004", binarize_qrels=1)
baseline.qrels = QrelsTransformer.drop_relevance(
    baseline.qrels, drop_values=999)

In [47]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), AreaUnderReceiver(),
              MeanAverageError(), LabelDistribution()],
    filter_qrels=True
)

In [48]:
res = meta_exp.evaluate()

In [49]:
metadata = read_metadata(BASE_DIR)

[topic_gen] [WARNING] (io.py:48) Metadata not found for result 2025-12-08_07:09:33, skipping...


In [50]:
def prompt_to_components(prompt: str) -> dict:
    parts = prompt.split("-")
    components = []
    if "title" not in parts:
        components.append("Title")
    if "description" not in parts:
        components.append("Description")
    if "narrative" not in parts:
        components.append("Narrative")
    if prompt == "-DNA-zero-shot":
        return "Title, Description, Narrative"
    return ", ".join(components)

In [51]:
df = pd.DataFrame(res)
missing = df.groupby("name")["missing_qrels"].max().to_dict()  # we take the max per group because label dist does not report missings yet

df = df.pivot(index="name", columns="measure", values="value").reset_index()
df = df.merge(metadata, left_on="name", right_on="date")
df["model"].unique()

df["missing"] = df["name"].map(missing)
df["missing"] = abs(df["missing"]- 311410 + 2951)

In [53]:
df.head()

,name,AreaUnderReceiver,CohenKappa,MeanAverageError,label_dist(0),label_dist(1),date,model,data,prompt,...,topics_model,topics_data,topics_prompt,topics_k,topics_nqueries,topics_ndocspos,topics_ndocsneg,topics_output,topics_task,missing
0,2025-11-26_11:17:24,0.812403,0.643923,0.169434,0.430701,0.569299,2025-11-26_11:17:24,gpt-4.1,robust,-DNA-zero-shot,...,human,robust,human,None,None,None,None,/workspaces/conf26-generating-topics/data/inte...,topics,0
1,2025-11-27_06:56:22,0.896110,0.758615,0.102730,0.284983,0.715017,2025-11-27_06:56:22,Qwen3-30B,robust,-DNA-zero-shot,...,human,robust,human,None,None,None,None,../data/interim/qrels-robust-reference,topics,21
2,2025-11-27_07:02:05,0.856400,0.684434,0.134537,0.287125,0.712875,2025-11-27_07:02:05,Qwen3-30B,robust,-DNA-zero-shot-masked-description-narrative,...,human,robust,human,None,None,None,None,../data/interim/qrels-robust-topics-masked,topics,15
3,2025-11-27_07:07:40,0.879704,0.727497,0.115962,0.285130,0.714870,2025-11-27_07:07:40,Qwen3-30B,robust,-DNA-zero-shot-masked-description,...,human,robust,human,None,None,None,None,../data/interim/qrels-robust-topics-masked,topics,19
4,2025-11-27_07:13:24,0.885318,0.731940,0.113497,0.278800,0.721200,2025-11-27_07:13:24,Qwen3-30B,robust,-DNA-zero-shot-masked-narrative,...,human,robust,human,None,None,None,None,../data/interim/qrels-robust-topics-masked,topics,17


In [ ]:
df = pd.DataFrame(res)
missing = df.groupby("name")["missing_qrels"].max().to_dict()  # we take the max per group because label dist does not report missings yet


df = df.pivot(index="name", columns="measure", values="value").reset_index()
df = df.merge(metadata, left_on="name", right_on="date")

df["missing"] = df["name"].map(missing)
df["missing"] = abs(df["missing"]- 311410 + 2951)

sorter_components = [
    "Title, Description, Narrative",
    "Title, Description",
    "Title, Narrative",
    "Description, Narrative",
    "Title",
    "Description",
    "Narrative"
]

sorter_models = [
    "gpt-4.1",
    'Llama3.1-8B',
    'Qwen3-14B',
    'Mistral3-14B',
    'GPT-OSS-20B',
    'Qwen3-30B-A3B-Instruct-2507-FP8',
    'Qwen3-30B',
    'Llama3.1-70B',
    'Qwen3-Next-80B',
    'GPT-OSS-120B',
    'GPT-OSS-120B-O',
    'Deepseek-V3.2',
    'Gemini-2.5-Flash',
    ]

df["components"] = df["prompt"].apply(prompt_to_components)
df["components"] = pd.Categorical(df["components"], sorter_components)

df["model"] = pd.Categorical(df["model"], sorter_models)
df.sort_values(by=["components", "model"])[
    ["model", "prompt", "components", "CohenKappa", "MeanAverageError",
        "AreaUnderReceiver", "missing", "label_dist(0)", "label_dist(1)"]
]

,model,prompt,components,CohenKappa,MeanAverageError,AreaUnderReceiver,missing,label_dist(0),label_dist(1)
0,gpt-4.1,-DNA-zero-shot,"Title, Description, Narrative",0.643923,0.169434,0.812403,0,0.430701,0.569299
10,Llama3.1-8B,-DNA-zero-shot,"Title, Description, Narrative",0.633280,0.160864,0.816116,1515,0.325905,0.674095
14,Qwen3-14B,-DNA-zero-shot,"Title, Description, Narrative",0.716263,0.130867,0.847650,32,0.388489,0.611511
9,GPT-OSS-20B,-DNA-zero-shot,"Title, Description, Narrative",0.640795,0.176690,0.815888,8,0.476724,0.523276
1,Qwen3-30B,-DNA-zero-shot,"Title, Description, Narrative",0.758615,0.102730,0.896110,21,0.284983,0.715017
12,Llama3.1-70B,-DNA-zero-shot,"Title, Description, Narrative",0.767102,0.103307,0.880054,18,0.338220,0.661780
22,Qwen3-Next-80B,-DNA-zero-shot,"Title, Description, Narrative",0.765613,0.105049,0.876379,0,0.351406,0.648594
8,GPT-OSS-120B,-DNA-zero-shot,"Title, Description, Narrative",0.706431,0.139605,0.842325,315,0.431715,0.568285
11,GPT-OSS-120B-O,-DNA-zero-shot,"Title, Description, Narrative",0.715921,0.135208,0.847353,0,0.431040,0.568960
13,Deepseek-V3.2,-DNA-zero-shot,"Title, Description, Narrative",0.721227,0.131696,0.849591,20,0.420334,0.579666


In [55]:
def to_latex_with_prompt_index(df: pd.DataFrame) -> pd.DataFrame:
    table = df.copy()
    table = table.round(2)
    table = table[[
        "model", "components", "CohenKappa", "MeanAverageError", "AreaUnderReceiver",
        "label_dist(0)", "label_dist(1)"]]
    table.rename(
        columns={
            "CohenKappa": "Cohen $\\kappa$",
            "MeanAverageError": "MAE",
            "AreaUnderReceiver": "AUC",
            "label_dist(0)": "0",
            "label_dist(1)": "1",
            "model": "Model",
            "components": "Components",
            "missing": "Missing",
        },
        inplace=True,
    )

    table = table.sort_values(
        ["Components", "Model"]).set_index(["Components", "Model"])

    formatters = {
        "Cohen $\\kappa$": lambda x: f"{x:.2f}",
        "MAE": lambda x: f"{x:.2f}",
        "AUC": lambda x: f"{x:.2f}",
        "0": lambda x: f"{x:.2f}",
        "1": lambda x: f"{x:.2f}",
    }

    print(table.to_latex(
        index=True,
        escape=False,
        formatters=formatters,
        caption="Agreement measures per Model and Prompt",
        label="tab:agreement_prompt_index",
    ))

In [56]:
to_latex_with_prompt_index(df)

\begin{table}
\caption{Agreement measures per Model and Prompt}
\label{tab:agreement_prompt_index}
\begin{tabular}{llrrrrr}
\toprule
 &  & Cohen $\kappa$ & MAE & AUC & 0 & 1 \\
Components & Model &  &  &  &  &  \\
\midrule
\multirow[t]{10}{*}{Title, Description, Narrative} & gpt-4.1 & 0.64 & 0.17 & 0.81 & 0.43 & 0.57 \\
 & Llama3.1-8B & 0.63 & 0.16 & 0.82 & 0.33 & 0.67 \\
 & Qwen3-14B & 0.72 & 0.13 & 0.85 & 0.39 & 0.61 \\
 & GPT-OSS-20B & 0.64 & 0.18 & 0.82 & 0.48 & 0.52 \\
 & Qwen3-30B & 0.76 & 0.10 & 0.90 & 0.28 & 0.72 \\
 & Llama3.1-70B & 0.77 & 0.10 & 0.88 & 0.34 & 0.66 \\
 & Qwen3-Next-80B & 0.77 & 0.11 & 0.88 & 0.35 & 0.65 \\
 & GPT-OSS-120B & 0.71 & 0.14 & 0.84 & 0.43 & 0.57 \\
 & GPT-OSS-120B-O & 0.72 & 0.14 & 0.85 & 0.43 & 0.57 \\
 & Deepseek-V3.2 & 0.72 & 0.13 & 0.85 & 0.42 & 0.58 \\
\cline{1-7}
\multirow[t]{3}{*}{Title, Description} & Qwen3-30B & 0.73 & 0.11 & 0.89 & 0.28 & 0.72 \\
 & Qwen3-Next-80B & 0.73 & 0.12 & 0.87 & 0.32 & 0.68 \\
 & GPT-OSS-120B-O & 0.68 & 0.15 & 0.83